<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    1.lab.1. Wisconsin Cancer Analysis
  </strong>
</div>

### Background

You will be working with the Wisconsin Diagnostic Cancer Dataset, which contains features computed from digitized images of fine needle aspirates (FNA) of tissue samples.

The dataset includes:
- 569 instances
- 30 numeric features computed from the cell nuclei present in the image
- Binary classification: **Malignant** or **Benign** diagnosis

### Description

Your task is to perform an exploratory data analysis using Python. You should use libraries such as `pandas`, `numpy`, and `matplotlib` to:

- Analyze the distribution of features
- Identify correlations between variables
- Explore relationships between features and diagnosis
- Create meaningful visualizations to support your findings

### Deliverables

- A **ipynb** (Jupyter Notebook) file called **wisconsin-cancer-analysis.ipynb** that contains your code and analysis.
- The **notebook** should be self-contained and include one paragraph summarizing the main findings of your analysis.
- Use the provided **wisconsin-cancer-analysis.ipynb** template as a starting point.

In [ ]:
from sklearn.datasets import load_breast_cancer

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Initial configuration

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

### Reading Dataset

Lets read the dataset using pandas.

In [ ]:
breast_cancer = load_breast_cancer(as_frame=True)
df = breast_cancer.data

df["diagnosis"] = breast_cancer.target

# Map diagnosis values (0/1) to labels (malignant/benign)
diagnosis_labels = df["diagnosis"].map(dict(enumerate(breast_cancer.target_names)))

print(breast_cancer.DESCR)
df.head()

### Analyzing the Data

1. **df.shape():** This function help us to understand the number of rows (observations) and columns (features) in the dataset. This gives an overview of the dataset's size and structure.

In [ ]:
df.shape

2. **df.info():** This function help us to understand the dataset by showing the number of records in each column, type of data, whether any values are missing and how much memory the dataset uses.

In [ ]:
df.info()

3. **df.nunique():** This function tells us how many unique values exist in each column which provides insight into the variety of data in each feature.

In [ ]:
df.nunique()

4. **df.describe().T**: This method give me a statistical summary of the DataFrame (Transpose) showing values like count, mean, standard deviation, minimum and quartiles for each numerical column. It helps in summarizing the central tendency and spread of the data.

In [ ]:
# Ignore the diagnostic column because it only has 0 or 1.

df.drop(columns=["diagnosis"]).describe().T

5. **df.columns.tolist():** This converts the column names of the DataFrame into a Python list making it easy to access and manipulate the column names.

In [ ]:
df.columns.tolist()

6. **df.isnull().sum():** This function help us to check the number of missing (null) values in each column of the dataset.

In [ ]:
df.isnull().sum()

7. **df.isna().sum():** This function help us for the same purpose as .isnull(), since both are equivalent and return the count of NaN or undefined values per column.

In [ ]:
df.isna().sum()

8. **sns.boxplot():** This function help us to visualize the distribution of numerical data and identify possible outliers.

In [ ]:
plt.figure(figsize=(16, 6))
sns.boxplot(data=df)
plt.xticks(rotation=90)
plt.show()

### Observations

- So far, the dataset has no missing values (null, NaN, or undefined), which is good. If there were missing values, we would need to decide whether to replace them or remove those records.
- All the variables are continuous numerical features, except for the diagnosis column, which is a categorical binary feature (0 or 1).
- Some variables in the dataset have very small ranges (for example, mean compactness = 0.019, smoothness error = 0.0017), while others have very large ranges (for example, worst area = 4254, mean area = 2501). This shows that the dataset has variables on very different scales. Because of this, normalization or standardization will be necessary before applying machine learning models.


# Exploratory Data Analysis (EDA)

We will follow these phases:

1. **Target variable analysis:** distribution of the diagnosis (benign vs. malignant).
2. **Univariate analysis:** distribution of individual features.
3. **Bivariate analysis:** relationship between features and the target variable (diagnosis).
4. **Correlation matrix and heatmap:** to identify multicollinearity and strong associations.
5. **Joint exploration with PairPlot:** to visualize patterns and separability across multiple features simultaneously.

## 1. Target Variable Analysis

**Explanation:**

* **df\["diagnosis\_name"] = df\["diagnosis"].map(...):** Creates a new column (diagnosis_name) by mapping the numerical values of the target (0, 1) to their corresponding labels (malignant, benign).
* **sns.countplot(x="diagnosis\_name", data=df):** Generates a bar plot showing the frequency of each diagnosis category.
* **plt.xlabel("Diagnosis"):** Sets the label for the x-axis to make the chart more readable.

In [ ]:
sns.countplot(x=diagnosis_labels)
plt.xlabel("Diagnosis")
plt.ylabel("Frecuency")
plt.title("Distribution of Diagnosis")
plt.show()

diagnosis_counts = diagnosis_labels.value_counts()
print(diagnosis_counts)

### Observations

- The chart shows a higher number of benign tumors (357) than malignant tumors (212).

## 2. Univariate Analysis: Distribution of Characteristics

**Explanation:**

  * **plt.subplot(2, 3, i):** Divides the figure into a grid of 2 rows and 3 columns, assigning each graph to a position.
  * **sns.histplot(df[characteristic], kde=True, bins=30):** Plots the histogram of each characteristic with a density curve to visualize the distribution.
  * **plt.tight_layout():** Optimizes the spacing between subplots to avoid overlap.

In [ ]:
# Define the grid layout
n_cols = 5
n_rows = 6

plt.figure(figsize=(20, 20))

for i, feature in enumerate(df.drop(columns=["diagnosis"]).columns, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(df[feature], kde=True, bins=30)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

### Observations

- Most characteristics show a right-skewed distribution rather than a normal distribution. Only a few variables, such as average texture and worst texture, show a distribution closer to a Gaussian shape.
- Some variables, such as “average texture” and “worst texture,” are clustered closer to a central value, showing a more symmetrical shape.
- Some variables, such as “compactness error” or “concavity error,” are concentrated near 0, indicating that most records have small values.

## 3. Bivariate Analysis: Relationship between Variables and Diagnosis

**Explanation:**

* Since the target variable **`diagnosis`** is **binary (0 = malignant, 1 = benign)**, using a scatterplot would only place the data points on two horizontal lines. 

* To better capture the **distribution of feature values across the two classes**, we used **boxplots**:

  * **sns.boxplot(...):**

    * `x=df["diagnosis"].map({0: "malignant", 1: "benign"})`: Places the two diagnostic groups on the X-axis.
    * `y=feature`: Places each variable on the Y-axis.

In [ ]:
# Define the grid layout
n_cols = 5
n_rows = 6

plt.figure(figsize=(20, 20))

for i, feature in enumerate(df.drop(columns=["diagnosis"]).columns, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.boxplot(x=diagnosis_labels, y=feature, data=df)
    plt.title(f"{feature} vs Diagnosis")
    plt.xlabel("Diagnosis")
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

### Observations

- Most of the mean and worst variables show higher values in malignant tumors compared to benign ones. This suggests that these features are strong candidates for classification, as they provide clear separation between the two classes.
- The error variables tend to show similar distributions for benign and malignant cases. This indicates that they may contribute less for classification compared to mean and worst features.
- Several boxplots reveal the presence of outliers in both malignant and benign groups.

## 4. Correlation Matrix and Heatmap

**Explanation:**

* **df.corr():** Calculates the correlation (Pearson coefficient) between the numerical variables in the DataFrame.
* sns.heatmap(...):
  * **cmap=‘coolwarm’:** Uses a palette that differentiates between high (positive) and low (negative) values.
  * **annot=True:** Places the numerical values in each cell.
  * **fmt=".2f":** Defines the numerical format.
  * **linewidths=0.5:** Sets the thickness of the dividing lines.

In [ ]:
matrix_correlation = df.corr()

plt.figure(figsize=(25, 25))
sns.heatmap(matrix_correlation, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix between Variables")
plt.show()

### Observations

* There is a **strong positive correlation** between the variables related to **area, perimeter, and radius** in the three groups (mean, error, and worst), which is to be expected since these measures are mathematically dependent.
* The variables **compactness, concavity, and concave points** also show **high positive correlations** with each other in the average, error, and worst groups.
* A **negative correlation** is observed between the “average” and “worst” versions of the diagnostic variable.
* Features such as **texture, smoothness, symmetry, and fractal dimension** (in the average, error, and worst groups) show **lower correlations** with most other variables, which could indicate that they provide more **independent information** and may contribute less discriminatory power to the classification.

## 5. Joint Exploration: PairPlot

**Explanation:**

- **sns.pairplot(...):** Creates a grid of scatterplots showing the relationship between each pair of selected variables.
  - **diag\_kind='kde':** Shows density plots (KDE) on the diagonal instead of histograms, making it easier to visualize the distribution of each variable.
  - **hue="diagnosis":** Colors the points according to the diagnosis (malignant vs benign), which helps to visually identify patterns and separability between the two groups.
  - **palette="Set1":** Assigns a distinct color palette for clarity.
  - **.assign(diagnosis=diagnosis\_labels):** Temporarily adds a column with the diagnosis labels (malignant/benign) so they can be used in the visualization without changing the original dataset.

In [ ]:
subset = [
    "mean area",
    "mean perimeter",
    "mean compactness",
    "mean concavity",
    "mean concave points",
]

sns.pairplot(
    df[subset].assign(diagnosis=diagnosis_labels),
    diag_kind="kde",
    hue="diagnosis",
    palette="Set1",
)
plt.suptitle("Pairplot of Selected Variables with Diagnosis", y=1.02)
plt.show()

In [ ]:
subset = [
    "worst area",
    "worst perimeter",
    "worst compactness",
    "worst concavity",
    "worst concave points",
]

sns.pairplot(
    df[subset].assign(diagnosis=diagnosis_labels),
    diag_kind="kde",
    hue="diagnosis",
    palette="Set1",
)
plt.suptitle("Pairplot of Selected Variables with Diagnosis", y=1.02)
plt.show()

### Observations

- On the diagonal (KDE), benign tumors (blue) show higher and narrower curves, indicating that their values tend to be concentrated in a narrow range and mostly at low values. In contrast, malignant tumors (red) have wider curves shifted to the right, reflecting that these cases reach higher values and are more widely distributed.
- When comparing the variables with each other, a clear separation between the groups can be observed: low values are mainly associated with benign tumors, while high values mostly correspond to malignant tumors. This reinforces the idea that characteristics such as area, perimeter, concavity, and concave points are decisive in distinguishing between the two diagnoses.

## Conclusions of the Exploratory Analysis

- The dataset has more benign tumors than malignant ones.
- Most features are not normally distributed, many are skewed, and some are concentrated near small values.
- Malignant tumors generally have higher values for features like **radius, perimeter, area, concavity, and compactness**, while benign tumors have lower values. 
- Error-related features (like texture error or smoothness error) do not differ much between benign and malignant cases, so they may not help much in classification.
- Some features are strongly correlated, especially area, perimeter, radius, compactness, concavity, and concave points. This means they provide similar information.
- Other features, such as texture, smoothness, symmetry, and fractal dimension, are more independent and may add extra information.
- Overall, malignant tumors show more variability and higher values, while benign tumors are more concentrated and lower. This clear difference makes it easier to separate the two classes.